# Nano-Tech Artist — 단계 1 학습 (Colab)

조건부 도형 생성 cGAN을 학습합니다. Cloud Press 리포지토리: https://github.com/choichoi3227-crypto/cloud-press

**순서**: 1) 리포지토리 클론 2) GPU 확인 3) Drive 마운트(체크포인트 저장용) 4) 데이터셋 생성 5) 학습 6) Hugging Face Hub 업로드

## 1. 리포지토리 클론

In [ ]:
!git clone https://github.com/choichoi3227-crypto/cloud-press.git
%cd cloud-press/ai-models/training/nano-tech-artist
!pip install -q -r requirements.txt

## 2. GPU 확인
런타임 → 런타임 유형 변경 → T4 GPU로 설정한 뒤 아래 셀을 실행하세요.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Google Drive 마운트 (체크포인트 저장용)
Colab 세션이 끊겨도 학습을 이어갈 수 있도록, 체크포인트는 반드시 Drive에 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/cloud-press/checkpoints/nano-tech-artist"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("체크포인트 저장 위치:", CHECKPOINT_DIR)

## 4. 합성 도형 데이터셋 생성
실제 이미지 데이터셋 없이, 코드로 도형을 그려서 데이터를 만듭니다. 조건(색상+도형)마다 균등하게 분포합니다.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/cloud-press/data/shapes"
!python generate_dataset.py --out {DATA_DIR} --count 6000 --size 64

## 5. 학습 실행
체크포인트가 이미 있으면 자동으로 이어서 학습합니다 (세션이 끊겼다가 다시 실행해도 안전).

In [ ]:
!python train.py --data {DATA_DIR} --checkpoint-dir {CHECKPOINT_DIR} --epochs 100 --batch-size 128

## 6. 생성 결과 확인

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from model import Generator, NOISE_DIM
from dataset import condition_to_onehot
from PIL import Image
import matplotlib.pyplot as plt

gen = Generator()
gen.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/generator_final.pt", map_location="cpu"))
gen.eval()

def tensor_to_pil(t):
    t = (t.clamp(-1, 1) + 1) / 2 * 255
    return Image.fromarray(t.permute(1, 2, 0).byte().numpy())

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
samples = [("red", "circle"), ("blue", "triangle"), ("green", "square"),
           ("yellow", "circle"), ("purple", "triangle"), ("black", "square")]
torch.manual_seed(0)
for ax, (color, shape) in zip(axes.flat, samples):
    cond = condition_to_onehot(color, shape).unsqueeze(0)
    noise = torch.randn(1, NOISE_DIM)
    with torch.no_grad():
        img = gen(noise, cond)[0]
    ax.imshow(tensor_to_pil(img))
    ax.set_title(f"{color} {shape}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Hugging Face Hub에 업로드
학습 결과가 만족스러우면 추론 서버가 사용할 수 있도록 업로드합니다.

먼저 https://huggingface.co/settings/tokens 에서 write 권한 토큰을 발급받으세요.

In [ ]:
from huggingface_hub import login, HfApi

login()  # 토큰 입력 프롬프트가 뜹니다

HF_REPO_ID = "<your-username>/nano-tech-artist"  # 실제 사용자명으로 변경
MODEL_VERSION = "v0.1.0"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj=f"{CHECKPOINT_DIR}/generator_final.pt",
    path_in_repo=f"nano-tech-artist-{MODEL_VERSION}.pt",
    repo_id=HF_REPO_ID,
)
print("업로드 완료:", HF_REPO_ID)